# Lab Assignment 7

In [142]:
import numpy as np
import matplotlib.pyplot as plt
from keras.datasets import mnist

In [143]:
np.random.seed(42)
epsilon = 1e-8

## Part 1

In [144]:
def generate_part1(N=3000):
    x1 = np.random.uniform(-2, 2, N)
    x2 = np.random.uniform(-2, 2, N)
    X = np.vstack([x1, x2]).T
    y = ((x1**2 + x2**2) > 1.5).astype(int).reshape(-1,1)
    return X, y

def split_data(X,y):
    idx = np.random.permutation(len(X))
    X, y = X[idx], y[idx]

    n_train = int(0.7*len(X))
    n_val   = int(0.85*len(X))

    return (X[:n_train],y[:n_train],
            X[n_train:n_val],y[n_train:n_val],
            X[n_val:],y[n_val:])

In [145]:
def sigmoid(z): return 1/(1+np.exp(-z))
def relu(z): return np.maximum(0,z)
def d_sigmoid(a): return a*(1-a)
def d_relu(z): return (z>0).astype(float)

In [146]:
def bce(y,yhat):
    yhat = np.clip(yhat,epsilon,1-epsilon)
    return -np.mean(y*np.log(yhat)+(1-y)*np.log(1-yhat))

def accuracy(y,yhat):
    yhat_bin = (yhat >= 0.5).astype(int)
    return np.mean(yhat_bin == y)

In [147]:
def init_layers(layers):
    params={}
    for l in range(1,len(layers)):
        params["W"+str(l)] = np.random.randn(layers[l-1],layers[l])*0.1
        params["b"+str(l)] = np.zeros((1,layers[l]))
    return params

In [148]:
def forward_dense(X,params,activation):
    cache={"A0":X}
    L=len(params)//2

    for l in range(1,L):
        Z=cache["A"+str(l-1)]@params["W"+str(l)]+params["b"+str(l)]
        A=sigmoid(Z) if activation=="sigmoid" else relu(Z)
        cache["Z"+str(l)],cache["A"+str(l)] = Z,A

    ZL=cache["A"+str(L-1)]@params["W"+str(L)]+params["b"+str(L)]
    AL=sigmoid(ZL)
    cache["Z"+str(L)],cache["A"+str(L)] = ZL,AL
    return AL,cache

In [149]:
def backward_dense(y,params,cache,activation):
    grads={}
    L=len(params)//2
    m=len(y)

    dZ=cache["A"+str(L)]-y

    for l in reversed(range(1,L+1)):
        grads["dW"+str(l)] = cache["A"+str(l-1)].T@dZ/m
        grads["db"+str(l)] = np.sum(dZ,axis=0,keepdims=True)/m

        if l>1:
            dA=dZ@params["W"+str(l)].T
            if activation=="sigmoid":
                dZ=dA*d_sigmoid(cache["A"+str(l-1)])
            else:
                dZ=dA*d_relu(cache["Z"+str(l-1)])
    return grads

In [150]:
def update_sgd(params,grads,lr):
    for k in params:
        params[k]-=lr*grads["d"+k]

def update_momentum(params,grads,vel,lr,beta=0.9):
    for k in params:
        vel[k]=beta*vel[k]+lr*grads["d"+k]
        params[k]-=vel[k]

In [151]:
def train_dense(X_train,y_train,X_val,y_val,layers,
                activation,optimizer,epochs=100,lr=0.01):

    params=init_layers(layers)
    vel={k:np.zeros_like(v) for k,v in params.items()}

    train_loss_hist=[]
    val_loss_hist=[]

    for ep in range(epochs):
        if ep == 0:
          print("Before update W1 mean:", np.mean(params["W1"]))

        yhat,cache=forward_dense(X_train,params,activation)
        grads=backward_dense(y_train,params,cache,activation)

        if optimizer=="sgd":
            update_sgd(params,grads,lr)
        else:
            update_momentum(params,grads,vel,lr)

        train_loss_hist.append(bce(y_train,yhat))
        val_loss_hist.append(bce(y_val,
                                 forward_dense(X_val,params,activation)[0]))
        if ep == 0:
          print("After update W1 mean:", np.mean(params["W1"]))

        if ep % 10 == 0:
          print(f"Epoch {ep}, Loss: {train_loss_hist[-1]:.4f}")

    return params,train_loss_hist,val_loss_hist

## Part 2

In [152]:
def load_mnist():
    (X_train,y_train),(X_test,y_test)=mnist.load_data()

    mask_train=(y_train==0)|(y_train==1)
    mask_test =(y_test==0)|(y_test==1)

    X=np.concatenate([X_train[mask_train],X_test[mask_test]])
    y=np.concatenate([y_train[mask_train],y_test[mask_test]])

    X=X/255.0
    y=y.reshape(-1,1)

    return split_data(X,y)

In [153]:
def conv_forward(X,K):
    N,H,W=X.shape
    F=K.shape[0]
    out=np.zeros((N,H-F+1,W-F+1))

    for n in range(N):
        for i in range(H-F+1):
            for j in range(W-F+1):
                out[n,i,j]=np.sum(X[n,i:i+F,j:j+F]*K)
    return out

In [154]:
def maxpool(X):
    N,H,W=X.shape
    out=np.zeros((N,H//2,W//2))

    for n in range(N):
        for i in range(0,H,2):
            for j in range(0,W,2):
                out[n,i//2,j//2]=np.max(X[n,i:i+2,j:j+2])
    return out

In [155]:
def dropout(A,p=0.5):
    mask=np.random.binomial(1,p,A.shape)
    return A*mask/p

## Part 3

In [156]:
def update_adam(params,grads,m,v,t,lr,
                beta1=0.9,beta2=0.999):
    for k in params:
        m[k]=beta1*m[k]+(1-beta1)*grads["d"+k]
        v[k]=beta2*v[k]+(1-beta2)*(grads["d"+k]**2)

        m_hat=m[k]/(1-beta1**t)
        v_hat=v[k]/(1-beta2**t)

        params[k]-=lr*m_hat/(np.sqrt(v_hat)+epsilon)

## Master Table

In [157]:
def print_master(results):
    print("\n"+"="*75)
    print(f"{'Model':<10}{'Depth':<8}{'Act':<10}"
          f"{'Opt':<10}{'Params':<10}"
          f"{'Train':<10}{'Val':<10}{'Test':<10}")
    print("="*75)

    for r in results:
        print(f"{r[0]:<10}{r[1]:<8}{r[2]:<10}"
              f"{r[3]:<10}{r[4]:<10}"
              f"{r[5]:<10.4f}{r[6]:<10.4f}{r[7]:<10.4f}")

    print("="*75)

## Main

In [158]:
if __name__=="__main__":

    master=[]

    # -------- PART 1 --------
    X,y=generate_part1()
    X_train,y_train,X_val,y_val,X_test,y_test=split_data(X,y)

    configs=[
        ([2,8,1],2),
        ([2,8,8,8,8,1],5),
        ([2,8,8,8,8,8,8,8,8,8,1],10)
    ]

    for layers,depth in configs:
        for activation in ["sigmoid","relu"]:
            for optimizer in ["sgd","momentum"]:

                params,_,_ = train_dense(
                    X_train,y_train,X_val,y_val,
                    layers,activation,optimizer,
                    epochs=1000,
                    lr=0.1
                )

                train_loss=bce(y_train,
                    forward_dense(X_train,params,activation)[0])
                val_loss=bce(y_val,
                    forward_dense(X_val,params,activation)[0])
                test_loss=bce(y_test,
                    forward_dense(X_test,params,activation)[0])

                train_acc=accuracy(y_train,
                    forward_dense(X_train,params,activation)[0])
                val_acc=accuracy(y_val,
                    forward_dense(X_val,params,activation)[0])
                test_acc=accuracy(y_test,
                    forward_dense(X_test,params,activation)[0])

                param_count=sum(p.size for p in params.values())

                master.append([
                    "Dense",depth,activation,optimizer,
                    param_count,
                    train_acc,val_acc,test_acc
                ])

    print_master(master)

Before update W1 mean: -0.03693240184838322
After update W1 mean: -0.03693280516640475
Epoch 0, Loss: 0.7389
Epoch 10, Loss: 0.6368
Epoch 20, Loss: 0.6144
Epoch 30, Loss: 0.6091
Epoch 40, Loss: 0.6078
Epoch 50, Loss: 0.6074
Epoch 60, Loss: 0.6073
Epoch 70, Loss: 0.6073
Epoch 80, Loss: 0.6073
Epoch 90, Loss: 0.6073
Epoch 100, Loss: 0.6073
Epoch 110, Loss: 0.6073
Epoch 120, Loss: 0.6073
Epoch 130, Loss: 0.6073
Epoch 140, Loss: 0.6073
Epoch 150, Loss: 0.6073
Epoch 160, Loss: 0.6073
Epoch 170, Loss: 0.6073
Epoch 180, Loss: 0.6073
Epoch 190, Loss: 0.6073
Epoch 200, Loss: 0.6073
Epoch 210, Loss: 0.6073
Epoch 220, Loss: 0.6073
Epoch 230, Loss: 0.6073
Epoch 240, Loss: 0.6073
Epoch 250, Loss: 0.6073
Epoch 260, Loss: 0.6073
Epoch 270, Loss: 0.6073
Epoch 280, Loss: 0.6073
Epoch 290, Loss: 0.6073
Epoch 300, Loss: 0.6073
Epoch 310, Loss: 0.6073
Epoch 320, Loss: 0.6073
Epoch 330, Loss: 0.6073
Epoch 340, Loss: 0.6073
Epoch 350, Loss: 0.6073
Epoch 360, Loss: 0.6073
Epoch 370, Loss: 0.6073
Epoch 380, L